# Patrons de conception

## Stratégie

Le code ci-dessous calcule le total d'une commande selon son mode de livraison, avec la forme classique du patron : une classe mère et une sous-classe par mode. Aucune de ces classes ne garde d'état.

Réécrivez-le sans classe :

- deux fonctions `livraison_standard(poids)` et `livraison_express(poids)`, qui renvoient les frais ;
- une fonction `total(prix, poids, livraison)`, qui reçoit l'une de ces deux fonctions.

Pour une commande de 30 € et de 2 kg, vous devez obtenir les mêmes sorties que le code d'origine : `35.0` et `44.0`. Ajoutez ensuite un mode `retrait_en_magasin`, sans frais, sans modifier `total` : le total de cette commande doit valoir `30.0`.

In [ ]:
class ModeLivraison:
  def frais(self, poids: float) -> float:
    raise NotImplementedError


class LivraisonStandard(ModeLivraison):
  def frais(self, poids: float) -> float:
    return 5.0


class LivraisonExpress(ModeLivraison):
  def frais(self, poids: float) -> float:
    return 10.0 + 2.0 * poids


class Commande:
  def __init__(self, prix: float, poids: float, mode: ModeLivraison) -> None:
    self.prix = prix
    self.poids = poids
    self.mode = mode

  def total(self) -> float:
    return self.prix + self.mode.frais(self.poids)


print(Commande(30.0, 2.0, LivraisonStandard()).total())
print(Commande(30.0, 2.0, LivraisonExpress()).total())

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une fonction de livraison prend le poids et renvoie les frais, comme la méthode `frais`. `total` appelle la fonction qu'il reçoit : `livraison(poids)`.

```python
def livraison_standard(poids: float) -> float:
  return ...


def livraison_express(poids: float) -> float:
  return ...


def total(prix: float, poids: float, livraison) -> float:
  return ...


print(total(30.0, 2.0, livraison_standard))
print(total(30.0, 2.0, livraison_express))
```

### Solution

In [ ]:
from collections.abc import Callable


def livraison_standard(poids: float) -> float:
  return 5.0


def livraison_express(poids: float) -> float:
  return 10.0 + 2.0 * poids


def retrait_en_magasin(poids: float) -> float:
  return 0.0


def total(prix: float, poids: float, livraison: Callable[[float], float]) -> float:
  return prix + livraison(poids)


print(total(30.0, 2.0, livraison_standard))
print(total(30.0, 2.0, livraison_express))
print(total(30.0, 2.0, retrait_en_magasin))

La classe mère et ses sous-classes disparaissent : `Callable[[float], float]` dit tout ce que `total` attend d'une stratégie. Une classe redevient utile quand la stratégie garde un état ou regroupe plusieurs méthodes.

## Adaptateur

La boutique valide ses commandes avec un objet `Paiement`, dont la méthode `paie(euros)` renvoie `True` si le paiement est accepté. Elle veut utiliser `CaisseTierce`, une bibliothèque qu'elle ne peut pas modifier et dont l'interface est différente : `encaisse(centimes, devise)` prend un nombre entier de centimes et renvoie un dictionnaire.

Écrivez une classe `AdaptateurCaisse`, qui reçoit une `CaisseTierce` et respecte le protocole `Paiement` : sa méthode `paie` convertit le montant en centimes, appelle `encaisse` avec la devise `"EUR"`, et renvoie `True` si le statut est `"accepte"`.

Avec une caisse de plafond `10_000`, validez la commande 1 de 19,99 €, puis la commande 2 de 150 €. Vous devez obtenir `Encaissement de 1999 centimes (EUR)` et `Commande 1 validée`, puis `Encaissement de 15000 centimes (EUR)` et `Commande 2 refusée`.

In [ ]:
from typing import Protocol


class Paiement(Protocol):
  def paie(self, euros: float) -> bool: ...


def valide_commande(numero: int, total: float, paiement: Paiement) -> None:
  if paiement.paie(total):
    print(f"Commande {numero} validée")
  else:
    print(f"Commande {numero} refusée")


class CaisseTierce:
  def __init__(self, plafond: int) -> None:
    self._plafond = plafond

  def encaisse(self, centimes: int, devise: str) -> dict[str, str]:
    print(f"Encaissement de {centimes} centimes ({devise})")
    statut = "accepte" if centimes <= self._plafond else "refuse"
    return {"statut": statut}

In [ ]:
# Votre code ici

### Conseils supplémentaires

Gardez la caisse dans un attribut. Pour la conversion, utilisez `round(euros * 100)` et non `int(euros * 100)` : `19.99 * 100` vaut `1998.9999999999998`, que `int` tronque à `1998`.

```python
class AdaptateurCaisse:
  def __init__(self, caisse: CaisseTierce) -> None:
    self._caisse = caisse

  def paie(self, euros: float) -> bool:
    reponse = self._caisse.encaisse(..., "EUR")
    return ...


caisse = CaisseTierce(10_000)
valide_commande(1, 19.99, AdaptateurCaisse(caisse))
valide_commande(2, 150.0, AdaptateurCaisse(caisse))
```

### Solution

In [ ]:
class AdaptateurCaisse:
  def __init__(self, caisse: CaisseTierce) -> None:
    self._caisse = caisse

  def paie(self, euros: float) -> bool:
    reponse = self._caisse.encaisse(round(euros * 100), "EUR")
    return reponse["statut"] == "accepte"


caisse = CaisseTierce(10_000)
valide_commande(1, 19.99, AdaptateurCaisse(caisse))
valide_commande(2, 150.0, AdaptateurCaisse(caisse))

`AdaptateurCaisse` n'hérite pas de `Paiement` : le protocole décrit ce que `valide_commande` attend, et un vérificateur de types contrôle que l'adaptateur le respecte. Ni `valide_commande` ni `CaisseTierce` n'ont changé.

## Observateur

Une boutique veut déclencher plusieurs actions quand une commande est payée (envoyer un reçu, mettre à jour le stock…), sans que le code de paiement connaisse ces actions.

Écrivez une classe `BusEvenements` avec deux méthodes :

- `abonner(evenement, fonction)`, qui abonne une fonction à un événement ;
- `publier(evenement, *args)`, qui appelle avec `args`, dans l'ordre d'abonnement, toutes les fonctions abonnées à cet événement.

Abonnez `envoie_recu` et `met_a_jour_stock` à l'événement `"commande_payee"`, puis publiez-le pour la commande 42 : les deux messages doivent s'afficher. Publier un événement sans abonné ne doit rien faire.

**Documentation utile**

- [`collections.defaultdict`](https://docs.python.org/fr/3/library/collections.html#collections.defaultdict)

In [ ]:
def envoie_recu(numero_commande: int) -> None:
  print(f"Reçu envoyé pour la commande {numero_commande}")


def met_a_jour_stock(numero_commande: int) -> None:
  print(f"Stock mis à jour pour la commande {numero_commande}")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Gardez les abonnés dans un dictionnaire qui associe à chaque événement la liste de ses fonctions : `defaultdict(list)` crée la liste au premier abonnement. Dans `publier`, `.get(evenement, [])` n'ajoute pas d'entrée pour un événement sans abonné. Vous devez obtenir `Reçu envoyé pour la commande 42`, puis `Stock mis à jour pour la commande 42`.

```python
from collections import defaultdict


class BusEvenements:
  def __init__(self) -> None:
    self._abonnes = defaultdict(list)

  def abonner(self, evenement, fonction) -> None: ...

  def publier(self, evenement, *args) -> None:
    for fonction in self._abonnes.get(evenement, []):
      ...


bus = BusEvenements()
bus.abonner("commande_payee", envoie_recu)
bus.abonner("commande_payee", met_a_jour_stock)
bus.publier("commande_payee", 42)
```

### Solution

In [ ]:
from collections import defaultdict
from collections.abc import Callable


class BusEvenements:
  def __init__(self) -> None:
    self._abonnes: defaultdict[str, list[Callable[..., None]]] = defaultdict(list)

  def abonner(self, evenement: str, fonction: Callable[..., None]) -> None:
    self._abonnes[evenement].append(fonction)

  def publier(self, evenement: str, *args: object) -> None:
    for fonction in self._abonnes.get(evenement, []):
      fonction(*args)


bus = BusEvenements()
bus.abonner("commande_payee", envoie_recu)
bus.abonner("commande_payee", met_a_jour_stock)
bus.publier("commande_payee", 42)
bus.publier("commande_annulee", 42)  # Aucun abonné : rien ne se passe

Le code qui publie l'événement ne dépend d'aucun abonné : on ajoute une action en abonnant une fonction, sans modifier le code de paiement. Le TP sur les décorateurs écrit le même mécanisme sous forme de décorateur, avec `@abonne("commande_validee")`.

## Itérateur

Les rayons de la boutique forment un arbre : chaque `Arbre` a une valeur et des enfants. Les exercices suivants écrivent plusieurs parcours de cet arbre, de sorte que le code qui le parcourt n'ait pas à connaître sa structure.

### Un générateur

Ajoutez à la classe `Arbre` ci-dessous une méthode `__iter__`, écrite comme un générateur, qui parcourt l'arbre en profondeur : la valeur d'un nœud, puis celles de ses enfants, dans l'ordre. Affichez ensuite `list(rayons)` : vous devez obtenir `['boutique', 'informatique', 'claviers', 'livres']`.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Arbre:
  valeur: str
  enfants: list["Arbre"] = field(default_factory=list)

  # Votre code ici


rayons = Arbre(
  "boutique", [Arbre("informatique", [Arbre("claviers")]), Arbre("livres")]
)

#### Conseils supplémentaires

`yield self.valeur` donne la valeur du nœud. `yield from enfant` donne toutes celles d'un enfant : il appelle le `__iter__` de cet enfant, et c'est là qu'est la récursion.

```python
from collections.abc import Iterator


def __iter__(self) -> Iterator[str]:
  yield ...
  for enfant in self.enfants:
    yield from ...
```

#### Solution

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass, field


@dataclass
class Arbre:
  valeur: str
  enfants: list["Arbre"] = field(default_factory=list)

  def __iter__(self) -> Iterator[str]:
    yield self.valeur
    for enfant in self.enfants:
      yield from enfant


rayons = Arbre(
  "boutique", [Arbre("informatique", [Arbre("claviers")]), Arbre("livres")]
)
print(list(rayons))

### Une pile explicite

Le générateur récursif ajoute un niveau d'appels par niveau de l'arbre : sur un arbre très profond, il dépasse la limite de récursion de Python, comme le montre la cellule suivante.

Écrivez une classe `ParcoursEnProfondeur`, qui fait le même parcours sans récursion :

- `__init__(self, arbre)` crée une pile, une liste qui contient la racine ;
- `__iter__` renvoie `self` ;
- `__next__` lève `StopIteration` si la pile est vide. Sinon, il retire le dernier nœud de la pile, y ajoute ses enfants, et renvoie sa valeur.

`list(ParcoursEnProfondeur(rayons))` doit donner le même résultat que `list(rayons)`, et `len(list(ParcoursEnProfondeur(profond)))` doit valoir `3001`.

In [ ]:
profond = Arbre("feuille")
for numero in range(3000):
  profond = Arbre(f"niveau {numero}", [profond])

try:
  list(profond)
except RecursionError:
  print("RecursionError : trop de niveaux pour le générateur récursif")

In [ ]:
# Votre code ici

#### Conseils supplémentaires

La pile rend d'abord le nœud ajouté en dernier : ajoutez les enfants du dernier au premier avec `reversed(noeud.enfants)`, pour que le premier enfant sorte le premier.

```python
class ParcoursEnProfondeur:
  def __init__(self, arbre: Arbre) -> None:
    self._pile = [arbre]

  def __iter__(self) -> "ParcoursEnProfondeur":
    return self

  def __next__(self) -> str:
    if not self._pile:
      raise ...
    noeud = self._pile.pop()
    self._pile.extend(...)
    return ...


print(list(ParcoursEnProfondeur(rayons)))
print(len(list(ParcoursEnProfondeur(profond))))
```

#### Solution

In [ ]:
class ParcoursEnProfondeur:
  def __init__(self, arbre: Arbre) -> None:
    self._pile = [arbre]

  def __iter__(self) -> "ParcoursEnProfondeur":
    return self

  def __next__(self) -> str:
    if not self._pile:
      raise StopIteration
    noeud = self._pile.pop()
    self._pile.extend(reversed(noeud.enfants))
    return noeud.valeur


print(list(ParcoursEnProfondeur(rayons)))
print(len(list(ParcoursEnProfondeur(profond))))

La position du parcours est dans la pile de l'itérateur, et non dans `Arbre` ou dans la pile d'appels de Python : c'est l'itérateur du GoF, et deux parcours d'un même arbre ne se gênent pas.

### Un second parcours (facultatif)

Écrivez une classe `ParcoursEnLargeur`, qui parcourt l'arbre niveau par niveau, avec une file `collections.deque` à la place de la pile. `list(ParcoursEnLargeur(rayons))` doit donner `['boutique', 'informatique', 'livres', 'claviers']`.

**Documentation utile**

- [`collections.deque`](https://docs.python.org/fr/3/library/collections.html#collections.deque)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`popleft()` retire le nœud le plus ancien de la file, et `extend` ajoute les enfants à sa fin, dans leur ordre.

```python
from collections import deque


class ParcoursEnLargeur:
  def __init__(self, arbre: Arbre) -> None:
    self._file = deque([arbre])

  def __iter__(self) -> "ParcoursEnLargeur":
    return self

  def __next__(self) -> str:
    if not self._file:
      raise StopIteration
    noeud = self._file.popleft()
    self._file.extend(...)
    return ...


print(list(ParcoursEnLargeur(rayons)))
```

#### Solution

In [ ]:
from collections import deque


class ParcoursEnLargeur:
  def __init__(self, arbre: Arbre) -> None:
    self._file = deque([arbre])

  def __iter__(self) -> "ParcoursEnLargeur":
    return self

  def __next__(self) -> str:
    if not self._file:
      raise StopIteration
    noeud = self._file.popleft()
    self._file.extend(noeud.enfants)
    return noeud.valeur


print(list(ParcoursEnLargeur(rayons)))

`Arbre` n'a pas changé : l'intérêt du patron est d'offrir plusieurs parcours d'une même structure, sans que le code qui les utilise connaisse cette structure.